# 🎯 [WEL03_M08] 오염물질 대량 유입 가상 시나리오 부하 테스트 및 경보 반응 시간 검증
### Google Colab 실습 환경 (장병 복지·지원 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/WEL03/wel03_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장병 복지·지원 (Level 2) |
| **미션 ID** | WEL03_M08 |
| **미션 주제** | 오염물질 대량 유입 가상 시나리오 부하 테스트 및 경보 반응 시간 검증 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
전장에서는 급수차, 정수 장비, 우물, 하천 취수원 등 급수원의 수질 이상을 빨리 알아채지 못하면 장병의 건강과 부대 활동에 바로 영향을 주지만, 센서 값은 잡음과 고장이 많아 오경보로 불필요하게 급수를 끊는 일도 생긴다. 러시아-우크라이나 전쟁 사례에서 학습 초점으로 삼은 전장 위생·건강 위험 관리를 반영해, 본 프로젝트는 가상 수질검사 데이터를 분석해 위험 신호를 데이터로 판단하는 과제를 다룬다. 교육용 가상 급수원 8곳의 pH, 탁도, 잔류염소, 중금속, 대장균 측정 자료를 코드로 만들어 스키마 정의, 센서 결측·이상치 정제, 기준치 Baseline 판정, 복합 오염 징후 분류 모델, 신뢰도 지수와 오경보 감축, 상황판 대시보드, 급수 중단 HITL 결재선, 대량 오염 부하 시험, AAR 임계치 재조정, 통합 패키지까지 단계별로 구현한다. 최종 목표는 수질 위험 신호를 근거와 함께 빠르게 제시하되, 실제 음용 판정과 급수 중단은 공식 기준과 의무(예방의학) 담당 확인, 지휘관 결심에 따르도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/wel03_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 결재 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/wel03_data_spec.md`, `docs/wel03_aar_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/wel03_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/wel03_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/wel03_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 오염물질 대량 유입 가상 시나리오 부하 테스트 및 경보 반응 시간 검증
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

오염물질 대량 유입 가상 시나리오 부하 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **오염물질이 대량 유입되는 가상 시나리오에서 파이프라인 부하 시험을 하고 경보 반응 시간과 Fail-safe 동작을 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **급수원을 8곳에서 48곳으로 늘려 트래픽을 500% 키우고 30곳에 오염이 동시에 유입되는 상황에서 수집·정제·판정 주기의 처리 시간, 시스템 중단, 데이터 유실, 메모리 증가, 경보 탐지와 반응 시간을 평시와 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/wel03_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건

- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%


- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료


- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 급수원을 8곳에서 48곳(트래픽 500% 증가)으로 늘리고 30곳에 오염물질이 동시에 유입되는 시나리오로 수집·판정 파이프라인 부하 시험을 수행해, 시스템 중단·데이터 유실·메모리 증가·경보 반응 시간과 과부하 시 기능 축소(Fail-safe) 동작을 검증한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 급수원 가상 수질 센서 데이터 생성기 (실제 부대·위치·검사 결과 없음)
# 가상 급수원 8곳 x 14일 x 1시간 간격. 실제 수질(true_*)·센서 측정값·오염 사건·센서 고장 정답을 함께 만든다.
# 기준치는 교육용 가정값이며, 실제 음용 판정은 공인 검사 결과와 의무(예방의학) 담당 기준에 따른다.
# ---------------------------------------------------------------------------
PARAMS = ["ph", "turbidity", "chlorine", "metal", "ecoli"]
SOURCES = [("W01", "water_truck", 1.0, 2.0), ("W02", "water_truck", 3.5, 1.2), ("W03", "purification_unit", 2.2, 4.1),
           ("W04", "purification_unit", 5.0, 3.3), ("W05", "well", 0.8, 5.6), ("W06", "well", 4.4, 6.0),
           ("W07", "river_intake", 6.3, 1.8), ("W08", "river_intake", 6.9, 4.9)]
BASE = {"water_truck": (7.2, 0.25, 0.75, 1.5), "purification_unit": (7.0, 0.20, 0.85, 1.2),
        "well": (6.8, 0.45, 0.50, 3.0), "river_intake": (7.3, 0.55, 0.60, 2.0)}   # pH, 탁도(NTU), 잔류염소(mg/L), 중금속(ug/L)
LIMITS = {"ph": (5.8, 8.5), "turbidity": (1.0, 2.0), "chlorine": (0.2, 0.1), "metal": (5.0, 10.0), "ecoli": (1, 1)}  # (주의, 위험) 가정 기준
EVENT_TYPES = {  # 오염 사건: 변수별 최대 변화량, 최대치 도달 시간(h)
    "sewage": ({"turbidity": 2.8, "chlorine": -0.5, "ecoli": 30, "ph": -0.3}, 4),
    "metal": ({"metal": 22.0, "ph": -0.5}, 4),
    "disinfect_fail": ({"chlorine": -0.8, "ecoli": 6}, 6),
    "chemical": ({"ph": 1.7, "turbidity": 1.0}, 3),
    "creeping_metal": ({"metal": 13.0}, 18),
}
FAULT_TYPES = ["stuck", "spike", "drift", "dropout"]
NOISE = {"ph": 0.04, "turbidity": 0.06, "chlorine": 0.03, "metal": 0.35}
DIGITS = {"ph": 2, "turbidity": 2, "chlorine": 2, "metal": 1}


def make_dataset(days=14, seed=7, n_events=6, n_faults=10, events=None, faults=None, sources=None):
    """events: [(급수원 번호, 사건 유형, 시작 시각 h, 지속 h, 강도)], faults: [(급수원 번호, 고장 유형, 항목, 시작 h, 지속 h)]"""
    rng = np.random.default_rng(seed)
    sources = sources or SOURCES
    T = days * 24
    tt = np.arange(T)
    hour = tt % 24
    night = (hour < 6) | (hour >= 20)
    rain = np.zeros(T)
    for _ in range(max(1, days // 4)):
        s0, d = int(rng.integers(0, T - 14)), int(rng.integers(6, 14))
        rain[s0:s0 + d] = np.maximum(rain[s0:s0 + d], rng.uniform(0.6, 1.0))
    if events is None:
        events = [(int(rng.integers(len(sources))), str(rng.choice(list(EVENT_TYPES))), int(rng.integers(min(24, T // 4), max(T // 4 + 1, T - 48))),
                   int(rng.integers(10, 30)), 1.0) for _ in range(n_events)]
    if faults is None:
        faults = [(int(rng.integers(len(sources))), str(rng.choice(FAULT_TYPES)), str(rng.choice(["ph", "turbidity", "chlorine", "metal"])),
                   int(rng.integers(12, max(13, T - 30))), int(rng.integers(6, 24))) for _ in range(n_faults)]
    out = []
    for si, (sid, stype, x, y) in enumerate(sources):
        ph0, tb0, cl0, mt0 = BASE[stype]
        tr = {"ph": ph0 + 0.08 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 0.03, T),
              "turbidity": tb0 * (1 + 0.1 * rng.normal(0, 1, T)),
              "chlorine": (cl0 * np.exp(-0.022 * hour) if stype == "water_truck" else cl0 + rng.normal(0, 0.03, T)),
              "metal": mt0 + rng.normal(0, 0.2, T), "ecoli": np.zeros(T)}
        if stype in ("river_intake", "well"):
            tr["turbidity"] = tr["turbidity"] + rain * (0.9 if stype == "river_intake" else 0.45)
        f_all, etype = np.zeros(T), np.array(["none"] * T, dtype=object)
        for (s_i, et, onset, dur, scale) in events:
            if s_i != si:
                continue
            delta, ramp = EVENT_TYPES[et]
            f = np.clip((tt - onset + 1) / ramp, 0, 1) * ((tt >= onset) & (tt < onset + dur + ramp))
            for k, v in delta.items():
                tr[k] = tr[k] + v * f * scale
            etype[f > 0] = et
            f_all = np.maximum(f_all, f)
        tr["chlorine"] = np.clip(tr["chlorine"], 0.02, None)
        tr["turbidity"] = np.clip(tr["turbidity"], 0.05, None)
        tr["metal"] = np.clip(tr["metal"], 0.1, None)
        grade = np.where(f_all >= 0.2, 2, 0)
        mild = (tr["turbidity"] > LIMITS["turbidity"][0]) | (tr["chlorine"] < LIMITS["chlorine"][0]) | (tr["metal"] > LIMITS["metal"][0])
        grade = np.where((grade == 0) & mild, 1, grade)
        ms = {}
        for p in ["ph", "turbidity", "chlorine", "metal"]:
            sd = NOISE[p] * np.where(night, 1.5, 1.0) * (tr[p] if p == "turbidity" else 1)
            ms[p] = np.round(np.clip(tr[p] + rng.normal(0, 1, T) * sd, 0, None), DIGITS[p])
        ms["ecoli"] = (rng.poisson(tr["ecoli"]) + (rng.random(T) < 0.004)).astype(float)
        fault, fparam = np.array(["none"] * T, dtype=object), np.array(["-"] * T, dtype=object)
        for (s_i, ft, p, s0, d) in faults:
            if s_i != si:
                continue
            seg = slice(s0, min(T, s0 + d))
            if ft == "stuck":
                ms[p][seg] = ms[p][s0]
            elif ft == "spike":
                seg = slice(s0, s0 + 1)
                ms[p][s0] = {"ph": 11.8, "turbidity": 38.0, "chlorine": 0.0, "metal": 75.0}[p]
            elif ft == "drift":
                ramp = np.linspace(0, 1, seg.stop - s0)
                ms[p][seg] = np.clip(ms[p][seg] + ramp * {"ph": -1.6, "turbidity": 2.6, "chlorine": -0.75, "metal": 12.0}[p], 0, None).round(DIGITS[p])
            elif ft == "dropout":
                for q in PARAMS:
                    ms[q][seg] = np.nan
                p = "all"
            fault[seg], fparam[seg] = ft, p
        df = pd.DataFrame({"source_id": sid, "source_type": stype, "x_km": x, "y_km": y, "t": tt,
                           "cond": np.where(rain > 0, "storm", np.where(night, "night", "day")),
                           **{p: ms[p] for p in PARAMS}, **{f"true_{p}": tr[p] for p in PARAMS},
                           "event": etype, "fault": fault, "fault_param": fparam, "grade": grade})
        out.append(df)
    return pd.concat(out, ignore_index=True)
# ---------------------------------------------------------------------------

# ---- 전처리 공통 함수: 물리 범위 점검 -> 고정값(stuck) 표시 -> 단발 스파이크 제거 -> 결측 보정 ----
PHYS = {"ph": (0, 14), "turbidity": (0, 1000), "chlorine": (0, 10), "metal": (0, 1000), "ecoli": (0, 100000)}
SPIKE_K = {"day": 6.0, "night": 7.0, "storm": 9.0}          # 상황별(주간/야간/악천후) 스파이크 판정 배수
MIN_JUMP = {"ph": 0.8, "turbidity": 1.5, "chlorine": 0.3, "metal": 6.0}
STUCK_RUN, SHORT_GAP = 6, 3


def clean(df, spike_k=None, min_jump=None):
    """급수원별 시계열을 정제하고 항목별 처리 내역(<항목>_flag: range/stuck/spike/interp/long_gap)을 남긴다."""
    spike_k = spike_k or SPIKE_K
    min_jump = MIN_JUMP if min_jump is None else min_jump
    out = df.sort_values(["source_id", "t"]).reset_index(drop=True)
    for sid, idx in out.groupby("source_id").groups.items():
        idx = np.asarray(idx)
        kk = out.loc[idx, "cond"].map(spike_k).to_numpy(float)
        for p in PARAMS:
            v = out.loc[idx, p].to_numpy(float).copy()
            fl = np.array([""] * len(v), dtype=object)
            lo, hi = PHYS[p]
            bad = (v < lo) | (v > hi)
            v[bad], fl[bad] = np.nan, "range"
            if p != "ecoli":
                chg = np.r_[True, v[1:] != v[:-1]]                       # 같은 값이 STUCK_RUN시간 이상 반복 = 고정값 고장 의심
                rid = np.cumsum(chg)
                rlen = np.bincount(rid)[rid]
                fl[(rlen >= STUCK_RUN) & ~chg & ~np.isnan(v)] = "stuck"
                d = np.abs(np.diff(v[~np.isnan(v)]))
                sig = 1.4826 * np.median(d) + 1e-6 if len(d) else 1.0
                up, dn = v - np.r_[np.nan, v[:-1]], v - np.r_[v[1:], np.nan]
                thr = np.maximum(kk * sig, min_jump[p])
                spike = (np.abs(up) > thr) & (np.abs(dn) > thr) & (np.sign(up) == np.sign(dn))   # 튀었다가 바로 돌아온 값만
                v[spike], fl[spike] = np.nan, "spike"
            s = pd.Series(v)
            na = s.isna().to_numpy()
            grp = np.cumsum(np.r_[True, na[1:] != na[:-1]])
            glen = pd.Series(na.astype(int)).groupby(grp).transform("sum").to_numpy()
            lin, ff = s.interpolate(limit_area="inside").to_numpy(), s.ffill().bfill().to_numpy()
            short, longg = na & (glen <= SHORT_GAP), na & (glen > SHORT_GAP)
            v = np.where(short, np.where(np.isnan(lin), ff, lin), np.where(longg, ff, v))
            fl[short & (fl == "")] = "interp"
            fl[longg] = "long_gap"
            out.loc[idx, p] = v
            out.loc[idx, f"{p}_flag"] = fl
    return out

# ---- 가정 기준치 판정(Baseline)과 시계열 특징 공통 함수 ----
GRADE_NAME = {0: "안전", 1: "주의", 2: "위험"}


def rule_grade(df):
    """기준치 1차 필터: 위험(음용 부적합 의심) = 한 항목이라도 위험 기준 초과, 주의 = 주의 기준 초과."""
    ph, tb, cl, mt, ec = (df[p].to_numpy(float) for p in PARAMS)
    danger = (ph < LIMITS["ph"][0]) | (ph > LIMITS["ph"][1]) | (tb > LIMITS["turbidity"][1]) | (cl < LIMITS["chlorine"][1]) \
        | (mt > LIMITS["metal"][1]) | (ec >= LIMITS["ecoli"][1])
    caution = (tb > LIMITS["turbidity"][0]) | (cl < LIMITS["chlorine"][0]) | (mt > LIMITS["metal"][0])
    return np.where(danger, 2, np.where(caution, 1, 0))


def reasons(row):
    r = []
    if not LIMITS["ph"][0] <= row["ph"] <= LIMITS["ph"][1]:
        r.append(f"pH {row['ph']:.2f}")
    if row["turbidity"] > LIMITS["turbidity"][0]:
        r.append(f"탁도 {row['turbidity']:.2f}NTU")
    if row["chlorine"] < LIMITS["chlorine"][0]:
        r.append(f"잔류염소 {row['chlorine']:.2f}mg/L")
    if row["metal"] > LIMITS["metal"][0]:
        r.append(f"중금속 {row['metal']:.1f}ug/L")
    if row["ecoli"] >= 1:
        r.append(f"대장균 {row['ecoli']:.0f}")
    return ", ".join(r) or "-"


FEATS = [f"{p}{s}" for p in ["ph", "turbidity", "chlorine", "metal"] for s in ["", "_d1", "_d3", "_dev", "_sd6"]] + \
        ["ecoli", "ecoli_3h", "storm", "night", "n_flag", "truck", "purif", "well", "river"]


def add_features(c):
    """시간 순서를 지키는(과거 값만 쓰는) 급수원별 특징: 변화량, 24시간 기준선 대비 편차, 6시간 변동성, 대장균 누적 등."""
    c = c.sort_values(["source_id", "t"]).reset_index(drop=True)
    g = c.groupby("source_id")
    for p in ["ph", "turbidity", "chlorine", "metal"]:
        c[f"{p}_d1"] = g[p].diff().fillna(0)
        c[f"{p}_d3"] = g[p].diff(3).fillna(0)
        base = g[p].transform(lambda s: s.shift(1).rolling(24, min_periods=4).median())
        c[f"{p}_dev"] = (c[p] - base).fillna(0)
        c[f"{p}_sd6"] = g[p].transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0)
    c["ecoli_3h"] = g["ecoli"].transform(lambda s: s.rolling(3, min_periods=1).sum())
    c["storm"], c["night"] = (c["cond"] == "storm").astype(int), (c["cond"] == "night").astype(int)
    flags = [f"{p}_flag" for p in PARAMS if f"{p}_flag" in c]
    c["n_flag"] = sum(c[f].isin(["stuck", "spike", "long_gap", "range"]).astype(int) for f in flags) if flags else 0
    for k, name in [("truck", "water_truck"), ("purif", "purification_unit"), ("well", "well"), ("river", "river_intake")]:
        c[k] = (c["source_type"] == name).astype(int)
    return c

BUDGET_MS = 300          # 1시간 주기 처리 예산(모의): 직전 주기가 예산을 넘으면 기능 축소 모드로 전환
FULL_EVERY = 12          # 기능 축소 중에도 12시간마다 전체 정제(지연 처리)를 수행


def make_sources(n, seed=0):
    rng = np.random.default_rng(seed)
    types = [s[1] for s in SOURCES]
    return [(f"W{i + 1:02d}", types[i % len(types)], float(round(rng.uniform(0, 30), 1)), float(round(rng.uniform(0, 30), 1))) for i in range(n)]


def lite_check(batch, last):
    """기능 축소 모드: 레코드 단위 범위 점검 -> 직전 정상값 유지 -> 기준치 판정(정제·특징 계산 생략)."""
    b = batch.copy()
    for p in PARAMS:
        v = pd.to_numeric(b[p], errors="coerce")
        lo, hi = PHYS[p]
        v = v.where(v.between(lo, hi))
        b[p] = v.fillna(b["source_id"].map(last.get(p, {}))).fillna(0 if p == "ecoli" else np.nan)
        last.setdefault(p, {}).update(b.dropna(subset=[p]).set_index("source_id")[p].to_dict())
    b = b.dropna(subset=["ph", "turbidity", "chlorine", "metal"])
    b["g"] = rule_grade(b)
    return b


def run(n_src, days, events, seed, mode_auto=True, hours=None):
    src = make_sources(n_src)
    data = make_dataset(days=days, seed=seed, events=events, faults=[], sources=src)
    rng = np.random.default_rng(seed)
    bad_idx = rng.choice(len(data), int(len(data) * 0.01), replace=False)     # 1% 손상 레코드 주입(문자·None)
    data[PARAMS] = data[PARAMS].astype(object)
    data.loc[bad_idx[: len(bad_idx) // 2], "turbidity"] = "ERR"
    data.loc[bad_idx[len(bad_idx) // 2:], "ph"] = None
    stored, quarantine, alerts, ticks, down, last = 0, 0, {}, [], 0, {}
    buf, mode, switched = [], "full", None
    mem = []
    for h in range(hours or days * 24):
        t0 = time.perf_counter()
        try:
            batch = data[data["t"] == h]
            if mode_auto and mode == "full" and ticks and ticks[-1] > BUDGET_MS:
                mode, switched = "lite", h                                   # 한 번 전환하면 부하 해소 확인 전까지 유지
            ok_mask = batch[["ph", "turbidity", "chlorine", "metal"]].apply(pd.to_numeric, errors="coerce").notna().all(axis=1)
            quarantine += int((~ok_mask).sum())
            if mode == "full":
                buf.append(batch[ok_mask].astype({p: float for p in PARAMS}))
                win = pd.concat(buf[-24:])
                c = clean(win)
                res = c[c["t"] == h].assign(g=lambda d: rule_grade(d))
            else:
                res = lite_check(batch, last)
                buf.append(batch[ok_mask].astype({p: float for p in PARAMS}))
                if h % FULL_EVERY == FULL_EVERY - 1:                          # 지연 전체 정제(기록 보정용)
                    clean(pd.concat(buf[-FULL_EVERY:]))
            buf = buf[-24:]
            stored += len(batch)
            for sid in res.loc[res["g"] == 2, "source_id"]:
                alerts.setdefault(sid, []).append(h)
        except Exception:
            down += 1
        ticks.append((time.perf_counter() - t0) * 1000)
        mem.append(sum(b.memory_usage(deep=True).sum() for b in buf) / 1e6)       # 보관 버퍼 메모리(MB)
        last = {p: last.get(p, {}) for p in PARAMS}
    react = []
    for e in events:
        hs = [h for h in alerts.get(src[e[0]][0], []) if h >= e[2]]
        if hs:
            react.append(hs[0] - e[2])
    return {"src": n_src, "recs": len(data), "stored": stored, "quarantine": quarantine, "down": down, "mode": mode if switched is None else f"lite({switched}h~)",
            "tick_avg": np.mean(ticks), "tick_max": np.max(ticks), "mem_first": np.mean(mem[24:48]) if len(mem) >= 48 else mem[-1], "mem_last": np.mean(mem[-24:]),
            "react": react, "detected": len(react), "n_events": len(events), "ticks": ticks}


days = 2
ev8 = [(i, et, 20 + i, 30, 1.0) for i, et in zip(range(5), ["sewage", "metal", "chemical", "disinfect_fail", "sewage"])]
ev48 = [(i, ["sewage", "metal", "chemical", "disinfect_fail"][i % 4], 20 + i % 6, 30, 1.0) for i in range(0, 48, 48 // 30 + 0)][:30]
t_all = time.perf_counter()
R = {"평시(8곳)": run(8, days, ev8, 1), "부하 500%(48곳, 30곳 동시 오염)": run(48, days, ev48, 2)}
print(f"[교육용가상자료] 부하 시험 {days}일 모의, 전체 실행 {time.perf_counter() - t_all:.1f}초")
rows = []
for k, r in R.items():
    loss = 1 - r["stored"] / r["recs"]
    rows.append({"구분": k, "레코드": r["recs"], "격리(손상)": r["quarantine"], "유실률": f"{loss:.1%}", "중단": r["down"],
                 "모드": r["mode"], "주기 평균ms": round(r["tick_avg"], 1), "주기 최대ms": round(r["tick_max"], 1),
                 "메모리MB(초기->말기)": f"{r['mem_first']:.1f}->{r['mem_last']:.1f}",
                 "경보 탐지": f"{r['detected']}/{r['n_events']}", "반응 평균h": round(float(np.mean(r["react"])), 1)})
print(pd.DataFrame(rows).set_index("구분").T.to_string())
S = R["부하 500%(48곳, 30곳 동시 오염)"]
print(f"\n부하 500%: 시스템 중단 {S['down']}건(목표 0건: {'달성' if S['down'] == 0 else '미달'}), "
      f"데이터 유실률 {1 - S['stored'] / S['recs']:.1%}(목표 0%: {'달성' if S['stored'] == S['recs'] else '미달'})")
print(f"메모리 말기/초기 비율 {S['mem_last'] / max(S['mem_first'], 1e-6):.2f} (버퍼 24시간 고정, 1.2 이하면 누수 없음), "
      f"경보 반응 최대 {max(S['react'])}시간")

# 기능 축소를 끄고 전체 정제를 강행하면 주기 시간이 얼마나 늘어나는지 (6시간만 측정)
F = run(48, 1, [], 3, mode_auto=False, hours=6)
print(f"기능 축소 미적용(전체 정제 강행) 주기 평균 {np.mean(F['ticks']):.0f}ms vs 기능 축소 {S['tick_avg']:.0f}ms "
      f"(예산 {BUDGET_MS}ms) -> 병목: 급수원별 정제 반복(clean 함수)")
print("Fail-safe 원칙: 손상 레코드는 버리지 않고 격리 보관, 판정은 직전 정상값으로 이어가며, 과부하 시 정제·특징 계산을 미루고 경보 판정은 유지")

plt.figure(figsize=(9, 3.5))
plt.plot(R["평시(8곳)"]["ticks"], label="normal (8 sources)")
plt.plot(S["ticks"], label="500% load (48 sources, degraded mode)")
plt.plot(F["ticks"], label="500% load, full mode forced (6 h)")
plt.axhline(BUDGET_MS, color="red", ls="--", lw=1, label="budget")
plt.xlabel("simulated hour"); plt.ylabel("processing ms per tick"); plt.legend(); plt.title("Stress test tick time (synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 부하 시험 2일 모의, 전체 실행 11.9초
구분               평시(8곳) 부하 500%(48곳, 30곳 동시 오염)
레코드                 384                    2304
격리(손상)                3                      23
유실률                0.0%                    0.0%
중단                    0                       0
모드                 full               lite(1h~)
주기 평균ms           127.1                    87.7
주기 최대ms           183.6                   742.7
메모리MB(초기->말기)  0.1->0.1                0.6->0.6
경보 탐지               5/5                   30/30
반응 평균h              0.8                     0.9

부하 500%: 시스템 중단 0건(목표 0건: 달성), 데이터 유실률 0.0%(목표 0%: 달성)
메모리 말기/초기 비율 1.00 (버퍼 24시간 고정, 1.2 이하면 누수 없음), 경보 반응 최대 2시간
기능 축소 미적용(전체 정제 강행) 주기 평균 808ms vs 기능 축소 88ms (예산 300ms) -> 병목: 급수원별 정제 반복(clean 함수)
Fail-safe 원칙: 손상 레코드는 버리지 않고 격리 보관, 판정은 직전 정상값으로 이어가며, 과부하 시 정제·특징 계산을 미루고 경보 판정은 유지
```
</details>
